In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

c:\Users\Arun\Desktop\nitfsds_0825\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Load embedding model (Hugging Face)
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\Arun\Desktop\nitfsds_0825\venv\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Arun\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 427.43it/s, Materializing param=poo

In [ ]:


df = pd.read_csv("medical_docs.csv")

documents = []
for _, row in df.iterrows():
    documents.append({
        "doc_id": row["doc_id"],
        "title": row["title"],
        "category": row["category"],
        "text": row["text"]
    })

print(documents[0])


{'doc_id': 1, 'title': 'Diabetes Mellitus', 'category': 'Endocrinology', 'text': 'Diabetes mellitus is a chronic metabolic disorder characterized by persistent hyperglycemia resulting from defects in insulin secretion, insulin action, or both. Type 1 diabetes is caused by autoimmune destruction of pancreatic beta cells, leading to absolute insulin deficiency. Type 2 diabetes results from insulin resistance combined with relative insulin deficiency. Common symptoms include polyuria, polydipsia, unexplained weight loss, fatigue, and blurred vision. Long-term complications include cardiovascular disease, neuropathy, nephropathy, and retinopathy. Management of diabetes involves lifestyle modifications, blood glucose monitoring, and pharmacological therapy.'}


In [3]:
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunks.append(" ".join(words[i:i + chunk_size]))
    return chunks


chunks = []
for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "text": chunk,
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "category": doc["category"]
        })

print("Total chunks:", len(chunks))


Total chunks: 5


In [6]:
import faiss

# 1. Load embedding model
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# 2. Prepare texts
texts = [c["text"] for c in chunks]

# 3. Convert text → embeddings
embeddings = embedder.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

# 4. Convert to float32 (FAISS requirement)
embeddings = np.array(embeddings).astype("float32")

# 5. Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)

# 6. Add embeddings to index
index.add(embeddings)

print("Indexed vectors:", index.ntotal)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 465.43it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]

Indexed vectors: 5


In [9]:
def retrieve(query, top_k=5):
    q_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(q_embedding, top_k)

    return [
        {
            "text": chunks[i]["text"],
            "score": float(scores[0][rank])
        }
        for rank, i in enumerate(indices[0])
    ]


query = "Which diseases increase the risk of coronary artery disease?"
retrieved_chunks = retrieve(query)


In [10]:
retrieved_chunks

[{'text': 'Coronary artery disease occurs due to atherosclerotic plaque buildup within the coronary arteries, leading to reduced blood flow to the heart muscle. Risk factors include hypertension, diabetes, smoking, and dyslipidemia. Management includes lifestyle modification, pharmacotherapy, and revascularization when indicated.',
  'score': 0.7556813955307007},
 {'text': 'Hypertension is a common cardiovascular condition defined by persistently elevated arterial blood pressure. It is a major risk factor for stroke, myocardial infarction, heart failure, and chronic kidney disease. Primary hypertension has no identifiable cause and accounts for most cases. Secondary hypertension results from underlying conditions such as renal disease or endocrine disorders. Treatment includes lifestyle changes and antihypertensive medications.',
  'score': 0.3828657865524292},
 {'text': 'Diabetes mellitus is a chronic metabolic disorder characterized by persistent hyperglycemia resulting from defects 

In [11]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs = [(query, c["text"]) for c in retrieved_chunks]
rerank_scores = reranker.predict(pairs)

for c, score in zip(retrieved_chunks, rerank_scores):
    c["rerank_score"] = float(score)

retrieved_chunks = sorted(
    retrieved_chunks,
    key=lambda x: x["rerank_score"],
    reverse=True
)


c:\Users\Arun\Desktop\nitfsds_0825\venv\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Arun\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 165.34it/s, Materializing param=class

In [12]:
# Prompt Construction (This Is RAG)

context = "\n\n".join(
    f"[Context {i+1}] {c['text']}"
    for i, c in enumerate(retrieved_chunks[:3])
)

prompt = f"""
You are a technical expert.

Use the context below to answer the question.

{context}

Question:
{query}

Answer:
""".strip()

prompt

'You are a technical expert.\n\nUse the context below to answer the question.\n\n[Context 1] Coronary artery disease occurs due to atherosclerotic plaque buildup within the coronary arteries, leading to reduced blood flow to the heart muscle. Risk factors include hypertension, diabetes, smoking, and dyslipidemia. Management includes lifestyle modification, pharmacotherapy, and revascularization when indicated.\n\n[Context 2] Hypertension is a common cardiovascular condition defined by persistently elevated arterial blood pressure. It is a major risk factor for stroke, myocardial infarction, heart failure, and chronic kidney disease. Primary hypertension has no identifiable cause and accounts for most cases. Secondary hypertension results from underlying conditions such as renal disease or endocrine disorders. Treatment includes lifestyle changes and antihypertensive medications.\n\n[Context 3] Diabetes mellitus is a chronic metabolic disorder characterized by persistent hyperglycemia r

In [1]:
# Load Hugging Face LLM

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


c:\Users\Arun\Desktop\nitfsds_0825\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.2,
        do_sample=True
    )

response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)
